# Naive Bayes: su primer clasificador, contado desde cero

**Ciencia de Datos, Sección A** · Sesión 7 · 13 de agosto de 2026

Notebook companion de la presentación. Corran cada celda con `Shift+Enter`.

Dependencias: `pip install numpy matplotlib scikit-learn`

Hoy no usamos `sklearn` hasta el final: primero escribimos el clasificador completo.

## 1. El problema: clasificar texto

Un texto entra, una etiqueta sale. Trabajaremos con reseñas cortas de restaurantes en español.

El corpus está escrito a mano aquí abajo: 16 reseñas etiquetadas para entrenar y 4 para probar.

In [ ]:
textos = [
    "el servicio fue excelente y el pepián delicioso",
    "muy buena atención, volvería sin duda",
    "la comida llegó rápido y bien caliente",
    "precio justo y porciones generosas",
    "me encantó el lugar, ambiente agradable",
    "el mesero fue muy amable con nosotros",
    "buen café de huehuetenango, recomendado",
    "todo estuvo rico y limpio",
    "la comida llegó fría y tarde",
    "pésimo servicio, esperamos una hora",
    "muy caro para lo poco que sirven",
    "el lugar estaba sucio y ruidoso",
    "no volvería, la atención fue terrible",
    "me cobraron de más y nadie respondió",
    "la sopa estaba salada y fea",
    "mala experiencia, todo lento",
]

etiquetas = ["positiva"] * 8 + ["negativa"] * 8

test_x = [
    "el pepián estaba rico y el servicio amable",
    "todo llegó frío y muy caro",
    "buena atención y precio justo",
    "esperamos una hora, pésimo",
]
test_y = ["positiva", "negativa", "positiva", "negativa"]

print(len(textos), len(etiquetas), len(test_x))

## 2. De Bayes al clasificador

$$P(c \mid x) = \frac{P(x \mid c)\,P(c)}{P(x)} \;\propto\; P(x \mid c)\,P(c)$$

El denominador $P(x)$ es igual para todas las clases, así que no cambia al ganador. La regla de decisión es:

$$\hat{c} = \arg\max_c \; P(c)\,P(x \mid c)$$

El **supuesto naive** hace tratable a $P(x \mid c)$:

$$P(x \mid c) \approx \prod_{i=1}^{n} P(w_i \mid c)$$

Es falso (ignora la negación y el orden) y aun así el orden entre clases suele sobrevivir.

## 3. De texto a números: bolsa de palabras

Tokenización mínima a propósito: `.lower()` para unificar mayúsculas, `.split()` para cortar por espacios.

In [ ]:
def tokenizar(texto):
    return texto.lower().split()

print(tokenizar("El servicio fue EXCELENTE"))

In [ ]:
ejemplo = ["comida rica y barata",
           "servicio lento y caro"]

vocab_ej = sorted({w for t in ejemplo for w in tokenizar(t)})
print(vocab_ej)

idx_ej = {w: i for i, w in enumerate(vocab_ej)}
print(idx_ej)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

AZUL, ROJO, GRIS, LINEA = "#3A6EA5", "#B04A2E", "#75808E", "#E0E4EA"

def eje_limpio(ax):
    ax.grid(color=LINEA, lw=0.6, alpha=0.7)
    ax.spines[["top", "right"]].set_visible(False)

M = np.zeros((len(ejemplo), len(vocab_ej)), dtype=int)
for f, t in enumerate(ejemplo):
    for w in tokenizar(t):
        M[f, idx_ej[w]] += 1

fig, ax = plt.subplots(figsize=(7, 2.0))
ax.imshow(M, cmap="Blues", vmin=0, vmax=M.max())
ax.set_xticks(range(len(vocab_ej)), vocab_ej, rotation=30, ha="right")
ax.set_yticks(range(len(ejemplo)), [f"doc {i + 1}" for i in range(len(ejemplo))])
for f in range(M.shape[0]):
    for j in range(M.shape[1]):
        ax.text(j, f, M[f, j], ha="center", va="center",
                color="white" if M[f, j] > 0 else GRIS, fontsize=9)
ax.set_title("La bolsa de palabras: cada texto se vuelve una fila de conteos")
plt.tight_layout()
plt.show()

# El orden de las palabras se perdió: solo quedaron los conteos.

## 4. Entrenar es contar (MLE puro)

Los estimadores de máxima verosimilitud de la sesión del 4 de agosto son, aquí, frecuencias relativas:

$$\hat{P}(c) = \frac{\#\text{docs de la clase } c}{\#\text{docs}} \qquad \hat{P}(w \mid c) = \frac{\text{conteo}(w, c)}{\text{total de palabras en } c}$$

No hay optimizador ni iteraciones: una sola pasada de conteo.

In [ ]:
import numpy as np

vocab = sorted({w for t in textos for w in tokenizar(t)})
idx = {w: i for i, w in enumerate(vocab)}
clases = sorted(set(etiquetas))

conteos = np.zeros((len(clases), len(vocab)))
priors = np.zeros(len(clases))

for texto, y in zip(textos, etiquetas):
    c = clases.index(y)
    priors[c] += 1
    for w in tokenizar(texto):
        conteos[c, idx[w]] += 1

print(clases)
print("vocabulario:", len(vocab), "palabras")
print("conteos:", conteos.shape)
print("priors:", priors / priors.sum())

In [ ]:
X_docs = np.zeros((len(textos), len(vocab)), dtype=int)
for f, t in enumerate(textos):
    for w in tokenizar(t):
        X_docs[f, idx[w]] += 1

pct_ceros = (X_docs == 0).mean()

fig, ax = plt.subplots(figsize=(10, 3.8))
ax.imshow(X_docs, cmap="Blues", aspect="auto", vmin=0, vmax=2)
ax.axhline(7.5, color=ROJO, lw=1.5)
ax.set_yticks(range(len(textos)),
              [f"{'pos' if y == 'positiva' else 'neg'} {i + 1}"
               for i, y in enumerate(etiquetas)], fontsize=7)
ax.set_xticks(range(len(vocab)), vocab, rotation=90, fontsize=6)
ax.set_title(f"Matriz documento-término: 16 reseñas × {len(vocab)} palabras; "
             f"{pct_ceros:.0%} de las celdas son cero")
plt.tight_layout()
plt.show()

# La línea roja separa las 8 positivas (arriba) de las 8 negativas.
# Esta dispersión (casi todo cero) es la norma en texto real.

In [ ]:
colores = {"positiva": AZUL, "negativa": ROJO}

fig, axes = plt.subplots(1, 2, figsize=(9, 3.4), layout="constrained")
for ax, nombre in zip(axes, ["positiva", "negativa"]):
    c = clases.index(nombre)
    top = np.argsort(conteos[c])[::-1][:10]
    pos_y = np.arange(10)
    ax.barh(pos_y, conteos[c][top], color=colores[nombre], alpha=0.85)
    ax.set_yticks(pos_y, [vocab[i] for i in top])
    ax.invert_yaxis()
    ax.set_title(f"clase {nombre}")
    ax.set_xlabel("conteo")
    eje_limpio(ax)
fig.suptitle("Entrenar es contar: las 10 palabras más frecuentes por clase")
plt.show()

# 'y', 'el', 'la' dominan en ambas clases: frecuente no es lo mismo que
# informativo. El ejercicio 3 busca las palabras que sí separan.

## 5. Dos arreglos indispensables

**Laplace.** Una palabra que nunca apareció en una clase le da probabilidad 0, y como el modelo multiplica, un solo cero veta a la clase entera. Sumamos $\alpha = 1$ a cada conteo:

$$\hat{P}(w \mid c) = \frac{\text{conteo}(w, c) + \alpha}{\text{total}(c) + \alpha|V|}$$

**Logaritmos.** Multiplicar 100 números del orden de $10^{-4}$ da $10^{-400}$: underflow. Como $\log$ es creciente, el argmax no cambia:

$$\hat{c} = \arg\max_c \; \log P(c) + \sum_i \log P(w_i \mid c)$$

In [ ]:
# El underflow no es teórico: veámoslo
p = 1e-4
producto = 1.0
for _ in range(100):
    producto *= p
print("producto:", producto)                  # 0.0 (underflow)
print("suma de logs:", 100 * np.log(p))       # perfectamente finito

In [ ]:
c_pos = clases.index("positiva")
total = conteos[c_pos].sum()

fig, ax = plt.subplots(figsize=(7, 3.2))
for a, color, ls in [(0.01, AZUL, "-"), (1.0, ROJO, "-"), (20.0, GRIS, "--")]:
    p = (conteos[c_pos] + a) / (total + a * len(vocab))
    ax.plot(np.sort(p)[::-1], color=color, ls=ls, lw=2, label=f"alpha = {a}")
ax.axhline(1 / len(vocab), color=GRIS, ls=":", lw=1.2, label="uniforme = 1/|V|")
ax.set_yscale("log")
ax.set_xlabel("palabras ordenadas de más a menos probables")
ax.set_ylabel("P(w | positiva), escala log")
ax.set_title("Laplace: alpha chico respeta los conteos; alpha grande aplana")
ax.legend(frameon=False)
eje_limpio(ax)
plt.tight_layout()
plt.show()

# Con alpha = 0.01 las palabras no vistas casi valen cero; con alpha = 20
# todas las palabras se parecen. Ese es el eje del ejercicio 1.

## 6. Implementación completa desde cero

Todo el clasificador: entrenar (contar) y predecir (sumar logs y tomar el máximo).

In [ ]:
def entrenar(textos, etiquetas, alpha=1.0):
    vocab = sorted({w for t in textos for w in tokenizar(t)})
    idx = {w: i for i, w in enumerate(vocab)}
    clases = sorted(set(etiquetas))
    conteos = np.zeros((len(clases), len(vocab)))
    priors = np.zeros(len(clases))
    for texto, y in zip(textos, etiquetas):
        c = clases.index(y)
        priors[c] += 1
        for w in tokenizar(texto):
            conteos[c, idx[w]] += 1
    suav = conteos + alpha
    denom = suav.sum(axis=1, keepdims=True)
    return (clases, idx, np.log(priors / priors.sum()), np.log(suav / denom))

In [ ]:
def puntajes(texto, modelo):
    clases, idx, log_prior, log_veros = modelo
    p = log_prior.copy()
    for w in tokenizar(texto):
        if w in idx:            # ignoramos palabras no vistas
            p += log_veros[:, idx[w]]
    return p

def predecir(texto, modelo):
    clases = modelo[0]
    return clases[int(np.argmax(puntajes(texto, modelo)))]

## 7. Probarlo y evaluarlo

In [ ]:
modelo = entrenar(textos, etiquetas, alpha=1.0)

print(predecir("la comida llegó fría y tarde", modelo))
print(predecir("muy buena atención, todo rico", modelo))
print(puntajes("muy buena atención, todo rico", modelo), modelo[0])

In [ ]:
frase = "muy buena atención, todo rico"
clases_m, idx_m, log_prior, log_veros = modelo
tokens = [w for w in tokenizar(frase) if w in idx_m]

acum = np.zeros((len(clases_m), len(tokens) + 1))
acum[:, 0] = log_prior
for j, w in enumerate(tokens):
    acum[:, j + 1] = acum[:, j] + log_veros[:, idx_m[w]]

colores = {"positiva": AZUL, "negativa": ROJO}
fig, ax = plt.subplots(figsize=(8, 3.4))
for ci, c in enumerate(clases_m):
    ax.plot(range(len(tokens) + 1), acum[ci], marker="o", lw=2,
            color=colores[c], label=c)
ax.set_xticks(range(len(tokens) + 1), ["(prior)"] + tokens)
ax.set_ylabel("log-puntaje acumulado")
ganadora = clases_m[int(np.argmax(acum[:, -1]))]
ax.set_title(f"El cálculo completo, palabra por palabra: gana '{ganadora}'")
ax.legend(frameon=False)
eje_limpio(ax)
plt.tight_layout()
plt.show()

# Cada palabra suma su log-verosimilitud a cada clase; la línea que
# termina arriba es el argmax. Esto ES la función puntajes().

In [ ]:
def accuracy(modelo, xs, ys):
    aciertos = [predecir(t, modelo) == y for t, y in zip(xs, ys)]
    return sum(aciertos) / len(aciertos)

print("accuracy test:", accuracy(modelo, test_x, test_y))
print("accuracy train:", accuracy(modelo, textos, etiquetas))

In [ ]:
i_pos = clases.index("positiva")
i_neg = clases.index("negativa")

todos_x = textos + test_x
todos_y = etiquetas + test_y
es_test = [False] * len(textos) + [True] * len(test_x)

margenes = np.array([puntajes(t, modelo)[i_pos] - puntajes(t, modelo)[i_neg]
                     for t in todos_x])
orden = np.argsort(margenes)

fig, ax = plt.subplots(figsize=(8, 4.2))
for fila, i in enumerate(orden):
    ax.barh(fila, margenes[i],
            color=AZUL if todos_y[i] == "positiva" else ROJO,
            alpha=0.85, hatch="//" if es_test[i] else None,
            edgecolor="white")
ax.axvline(0, color=GRIS, lw=1.2)
ax.set_yticks([])
ax.set_xlabel("log P(positiva | texto) - log P(negativa | texto)")
ax.set_title("Margen por reseña: el signo decide (rayadas = test, color = etiqueta real)")
eje_limpio(ax)
plt.tight_layout()
plt.show()

# Barras azules a la derecha y rojas a la izquierda = todo bien
# clasificado. Las rayadas (test) tienen márgenes más cortos: el
# modelo está menos seguro fuera de lo que memorizó.

## 8. Lo mismo con scikit-learn

`CountVectorizer` es la bolsa de palabras; `MultinomialNB` es el conteo con suavizado. Escribimos las 30 líneas primero para que estas 5 no sean magia.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

vec = CountVectorizer()
X = vec.fit_transform(textos)
nb = MultinomialNB(alpha=1.0).fit(X, etiquetas)

print(nb.predict(vec.transform(test_x)))
print(test_y)
print("accuracy sklearn:", nb.score(vec.transform(test_x), test_y))

## 9. Ejercicios

Completen donde dice `# ¿Qué va aquí?`.

### Ejercicio 1: la perilla del suavizado

Agreguen un $\alpha$ ajustable y prueben varios valores. Pista: con $\alpha$ enorme, todos los conteos se parecen entre sí.

In [ ]:
resultados = []

for a in [0.01, 0.1, 1.0, 5.0, 20.0]:
    m = entrenar(textos, etiquetas, alpha=a)
    # ¿Qué va aquí?
    # Calcular accuracy sobre (test_x, test_y) y guardarla junto con a
    pass

# ¿Qué alpha da el mejor resultado?
# ¿Qué le pasa al modelo con alpha muy grande?
# Verificación (descomenten al terminar):
# print(resultados)

### Ejercicio 2: tres frases nuevas

Clasifiquen tres frases que el modelo nunca vio. La tercera es la trampa: ¿el modelo entiende el "no"?

In [ ]:
nuevas = [
    "el pepián estaba delicioso y bien servido",
    "esperamos cuarenta minutos por un café frío",
    "no me gustó nada la atención",
]

for frase in nuevas:
    # ¿Qué va aquí?
    # a) imprimir la clase predicha
    # b) imprimir los puntajes de cada clase
    pass

### Ejercicio 3: palabras más informativas

Un modelo así de simple se puede **leer**: las palabras más informativas son las de mayor diferencia de log-verosimilitud entre clases.

In [ ]:
def mas_informativas(modelo, k=5):
    clases, idx, log_prior, log_veros = modelo
    inv = {i: w for w, i in idx.items()}
    # ¿Qué va aquí?
    # 1) ratio = log_veros[1] - log_veros[0]
    # 2) ordenar con np.argsort
    # 3) devolver las k palabras de cada extremo
    pass

# Verificación (descomenten al terminar): ¿tienen sentido las palabras?
# print(mas_informativas(modelo, k=5))

## Lo esencial de hoy

- Clasificar es un **argmax**: $P(c)\,P(x \mid c)$, sin denominador
- El supuesto **naive** es falso y aun así el orden entre clases sobrevive
- **Bolsa de palabras**: el texto se vuelve conteos, se pierde el orden
- **Entrenar es contar**: los estimadores son MLE puro
- **Laplace** evita los ceros; los **logs** evitan el underflow
- Escribimos el clasificador completo antes de tocar `sklearn`

**Próxima clase (martes 18): Regresión Lineal desde la perspectiva MLE.** Ese día se asigna la HDT 3. Recordatorio: la HDT 2 se entrega hoy, jueves 13, a las 23:59.